In [1]:
# CELL 1: SETUP & CHECK
import os
import torch
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings

/Users/bhanuprakashbhat/auto-auditor-rag-agent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load Keys
load_dotenv()

# Check Device 
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f" Running on Device: {device.upper()}")

# Test Hugging Face Loading
print("📥 Testing Model Load (all-mpnet-base-v2)...")
hf = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={'device': 'cpu'} # Keep CPU for stability on first run
)
print(" Success! Model loaded locally.")

 Running on Device: MPS
📥 Testing Model Load (all-mpnet-base-v2)...
 Success! Model loaded locally.


In [3]:
# CELL 2.1: LOAD PDFs
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

DATA_PATH = "../data/rules/"

print(f"📂 Scanning {DATA_PATH}...")

# 1. Define the Loader
# This looks for any file ending in .pdf and uses PyPDFLoader to read it
loader = DirectoryLoader(DATA_PATH, glob="*.pdf", loader_cls=PyPDFLoader)

# 2. Load the content
docs = loader.load()

print(f"✅ Success! Loaded {len(docs)} pages of documents.")
print(f"Example Source: {docs[0].metadata['source']}")

📂 Scanning ../data/rules/...


Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 116 0 (offset 0)
Ignoring wrong pointing object 121 0 (offset 0)
Ignoring wrong pointing object 144 0 (offset 0)
Ignoring wrong pointing object 188 0 (offset 0)
Ignoring wrong pointing object 418 0 (offset 0)
Ignoring wrong pointing object 1256 0 (offset 0)
Ignoring wrong pointing object 1287 0 (offset 0)
Ignoring wrong pointing object 1296 0 (offset 0)


✅ Success! Loaded 172 pages of documents.
Example Source: ../data/rules/tax_incentives_for_early_stage_investors_Web_b47a3095-ceb9-4912-8155-a980cbb9c827.pdf


In [4]:
# CELL 2.2: SPLIT TEXT (FIXED)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Configure the Splitter
# We use r"" for the regex pattern to avoid SyntaxWarning
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", r"(?<=\. )", " ", ""] 
)

# 2. Split the documents
splits = text_splitter.split_documents(docs)

print(f"✅ Created {len(splits)} semantic chunks.")

# 3. Inspect a chunk
print("\n--- SAMPLE CHUNK 0 ---")
print(splits[0].page_content)

✅ Created 830 semantic chunks.

--- SAMPLE CHUNK 0 ---
Print whole section
Tax incentives for early stage
investors
Investors who purchased new shares in a qualifying early
stage innovation company may be eligible for tax
incentives.
Last updated 16 August 2024
About the tax incentives for early stage
investors
Check if you're eligible for tax incentives for early stage
investors.
Qualifying for the tax incentives
Check if you qualify for the tax incentives.
The sophisticated investor test


In [5]:
# CELL 2.3: BUILD VECTOR STORE
from langchain_community.vectorstores import Chroma

# Ensure you have the 'hf' model loaded from Cell 1
if 'hf' not in globals():
    print(" Error: 'hf' model not found. Please run Cell 1 first!")
else:
    print(" Embedding chunks... (This runs on your Mac's Neural Engine)")
    
    # This creates the folder 'chroma_db_hf' on your computer
    vectorstore = Chroma.from_documents(
        documents=splits, 
        embedding=hf, 
        persist_directory="../chroma_db_hf"
    )
    
    print("✅ Vector Index successfully saved to disk.")

 Embedding chunks... (This runs on your Mac's Neural Engine)
✅ Vector Index successfully saved to disk.


In [6]:
# CELL 2.4: BUILD KEYWORD INDEX
import pickle
from langchain_community.retrievers import BM25Retriever

BM25_PATH = "../bm25_retriever_hf.pkl"

print(" Building BM25 Index...")

# 1. Create the Retriever
bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 5

# 2. Save it (Pickle it)
with open(BM25_PATH, "wb") as f:
    pickle.dump(bm25_retriever, f)
    
print(f" Keyword Index saved to {BM25_PATH}")

 Building BM25 Index...
 Keyword Index saved to ../bm25_retriever_hf.pkl


In [7]:
# CELL 3: TEST RETRIEVAL (FAIL-SAFE VERSION)
import pickle
from langchain_community.vectorstores import Chroma

# --- IMPORT FIX & FAIL-SAFE ---
try:
    # correct location
    from langchain.retrievers import EnsembleRetriever
    print("✅ Imported EnsembleRetriever from langchain.retrievers")
except ImportError:
    try:
        from langchain_community.retrievers import EnsembleRetriever
        print("✅ Imported EnsembleRetriever from langchain_community")
    except ImportError:
        print("⚠️ library import failed. Using manual definition.")
        # Manual Fallback if library is acting up
        from langchain_core.retrievers import BaseRetriever
        from typing import List
        from langchain_core.callbacks import CallbackManagerForRetrieverRun
        from langchain_core.documents import Document

        class EnsembleRetriever(BaseRetriever):
            retrievers: List[BaseRetriever]
            weights: List[float]

            def _get_relevant_documents(
                self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
            ) -> List[Document]:
                # Simple aggregation strategy
                all_docs = []
                for retriever in self.retrievers:
                    all_docs.extend(retriever.invoke(query))
                # Deduplicate by content
                unique_docs = {d.page_content: d for d in all_docs}
                return list(unique_docs.values())[:5]

# --- MAIN LOGIC ---

# 1. Load Vector DB
# Note: passing 'hf' (HuggingFaceEmbeddings) is crucial here
if 'hf' not in globals(): 
    print("❌ ERROR: Please run Cell 1 to load 'hf' model first.")
else:
    vectorstore = Chroma(persist_directory="../chroma_db_hf", embedding_function=hf)
    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

    # 2. Load Keyword DB
    with open(BM25_PATH, "rb") as f:
        bm25_retriever = pickle.load(f)

    # 3. Hybrid Search
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=[0.4, 0.6]
    )

    # 4. Test Query
    query = "Is bug fixing eligible for R&D tax offset?"
    print(f"\n❓ Query: {query}\n")

    docs = ensemble_retriever.invoke(query)

    for i, doc in enumerate(docs):
        print(f"--- Result {i+1} ---")
        print(doc.page_content[:300] + "...") 
        print(f"[Source: {doc.metadata.get('source', 'Unknown')}]\n")

⚠️ library import failed. Using manual definition.

❓ Query: Is bug fixing eligible for R&D tax offset?



/var/folders/qh/48t3z34132lfc6chv_82jz2m0000gn/T/ipykernel_97840/3724482932.py:44: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory="../chroma_db_hf", embedding_function=hf)


--- Result 1 ---
generating new knowledge (involve scientific or technical uncertainty). While it will 
depend on the facts in each case, examples of activities that are not eligible as core 
R&D activities because they generally are not being conducted as experiments to test a 
hypothesis are: 
 bug testing (ident...
[Source: ../data/rules/RDTI Software activities and the RDTI PDF.pdf]

--- Result 2 ---
because you expect to claim the early investor tax offset, you may
wish to correct your PAYG instalments.
Can a member of a trust or partnership be
eligible for the early stage investor tax
offset?
A member of a trust (a beneficiary, unit-holder or object) or
partnership (a partner) may be entitled ...
[Source: ../data/rules/tax_incentives_for_early_stage_investors_Web_b47a3095-ceb9-4912-8155-a980cbb9c827.pdf]

--- Result 3 ---
QC48899
On this page
What if the company has a ruling on the tests
How I claim the early stage investor tax offset
How the tax offset affects PAYG income tax i

In [8]:
# CELL 4: DEFINE AGENT NODES (FIXED for LangChain v0.3)
from typing import TypedDict, List
from langchain_core.prompts import ChatPromptTemplate
# FIXED IMPORT: Use standard pydantic instead of langchain_core.pydantic_v1
from pydantic import BaseModel, Field 
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1. Define the State (The Agent's Short-term Memory)
class GraphState(TypedDict):
    question: str
    generation: str
    documents: List[str]
    web_search: str 

# 2. NODE: RETRIEVE
def retrieve(state):
    print("--- 🔍 RETRIEVING DOCUMENTS ---")
    question = state["question"]
    # We use the 'ensemble_retriever' we built in Cell 3
    documents = ensemble_retriever.invoke(question)
    return {"documents": documents, "question": question}

# 3. NODE: GRADE (Self-Correction)
def grade_documents(state):
    print("--- ⚖️ GRADING RELEVANCE ---")
    question = state["question"]
    documents = state["documents"]
    
    # Simple Grader LLM
    llm = ChatOpenAI(model="gpt-4o", temperature=0)
    
    # Structured output to force a clear "yes/no"
    class Grade(BaseModel):
        binary_score: str = Field(description="'yes' or 'no'")

    structured_llm_grader = llm.with_structured_output(Grade)
    system = "You are a strict grader. Does the document contain Australian Tax rules or definitions relevant to the user's question?"
    grade_prompt = ChatPromptTemplate.from_messages([("system", system), ("human", "Doc: {document} \n Question: {question}")])
    grader = grade_prompt | structured_llm_grader
    
    filtered_docs = []
    rewrite = "No"
    
    for d in documents:
        score = grader.invoke({"question": question, "document": d.page_content})
        if score.binary_score == "yes":
            filtered_docs.append(d)
            
    # If we filtered out ALL docs, we need to try searching again
    if not filtered_docs:
        print("   ❌ No relevant docs found. Triggering Query Rewrite.")
        rewrite = "Yes" 
    else:
        print(f"   ✅ Kept {len(filtered_docs)} relevant documents.")
        
    return {"documents": filtered_docs, "question": question, "web_search": rewrite}

# 4. NODE: TRANSFORM QUERY (The "Smart" Part)
def transform_query(state):
    print("--- 🔄 REWRITING QUERY ---")
    question = state["question"]
    llm = ChatOpenAI(model="gpt-4o", temperature=0)
    msg = [("human", f"The previous search for '{question}' failed. \n Write a better search query to find R&D Tax Incentive rules in Australia. Output ONLY the query.")]
    better_question = llm.invoke(msg).content
    print(f"   New Query: {better_question}")
    return {"question": better_question}

# 5. NODE: GENERATE
def generate(state):
    print("--- ✍️ GENERATING ANSWER ---")
    question = state["question"]
    documents = state["documents"]
    llm = ChatOpenAI(model="gpt-4o", temperature=0)
    
    prompt = ChatPromptTemplate.from_template(
        "You are an expert Australian R&D Tax Consultant. Use the context to answer. \n"
        "Cite the specific pdf source if possible. \n"
        "If the activity is excluded, warn the user explicitly. \n"
        "Context: {context} \n"
        "Question: {question} \n"
        "Answer:"
    )
    chain = prompt | llm | StrOutputParser()
    generation = chain.invoke({"context": documents, "question": question})
    return {"generation": generation}

In [9]:
# CELL 5: BUILD GRAPH
from langgraph.graph import END, StateGraph

def decide_next_step(state):
    if state["web_search"] == "Yes":
        return "transform_query" # Loop back
    return "generate" # Move forward

# 1. Initialize
workflow = StateGraph(GraphState)

# 2. Add Nodes
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("transform_query", transform_query)

# 3. Add Edges (The Flow)
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")

# Conditional Edge: Check grade before generating
workflow.add_conditional_edges(
    "grade_documents", 
    decide_next_step, 
    {
        "transform_query": "transform_query",
        "generate": "generate"
    }
)
workflow.add_edge("transform_query", "retrieve") # The Loop
workflow.add_edge("generate", END)

# 4. Compile
app = workflow.compile()
print("✅ Agent Compiled and Ready.")

✅ Agent Compiled and Ready.


In [10]:
# CELL 6: RUN THE AGENT
inputs = {"question": "Is bug fixing eligible for the R&D tax offset?"}

print(f"🚀 STARTING AGENT run for: '{inputs['question']}'\n")

# Stream the steps so you can watch it think
for output in app.stream(inputs):
    pass 

# Get final answer
final_state = app.invoke(inputs)
print("\n📝 FINAL ADVICE:")
print("-" * 50)
print(final_state["generation"])
print("-" * 50)

🚀 STARTING AGENT run for: 'Is bug fixing eligible for the R&D tax offset?'

--- 🔍 RETRIEVING DOCUMENTS ---
--- ⚖️ GRADING RELEVANCE ---
   ✅ Kept 1 relevant documents.
--- ✍️ GENERATING ANSWER ---
--- 🔍 RETRIEVING DOCUMENTS ---
--- ⚖️ GRADING RELEVANCE ---
   ✅ Kept 1 relevant documents.
--- ✍️ GENERATING ANSWER ---

📝 FINAL ADVICE:
--------------------------------------------------
Bug fixing is not eligible for the R&D tax offset. According to the document titled "RDTI guidance software activities and the RDTI," activities such as bug testing, which involves identifying and fixing errors in code, are not considered eligible as core R&D activities. This is because they generally do not involve conducting experiments to test a hypothesis or generate new knowledge involving scientific or technical uncertainty. (Source: RDTI Software activities and the RDTI PDF, page 10)
--------------------------------------------------
